# CUDA C++ GPU Lab — Kaggle Notebook

Run cells in order after setting **Notebook Settings → Accelerator = GPU**. This notebook covers baseline CUDA execution and memory optimization.

## 1. Verify NVIDIA GPU and CUDA compiler

In [ ]:
!nvidia-smi
!nvcc --version
!which nvcc

If `nvcc` is missing, this session does not expose the CUDA C++ compiler. A CUDA-enabled PyTorch package does not necessarily include `nvcc`.

## 2. Clone the repository or pull latest main

In [ ]:
from pathlib import Path
import os

repo_dir = Path('/kaggle/working/cuda-cpp-gpu-lab')
if repo_dir.exists():
    %cd /kaggle/working/cuda-cpp-gpu-lab
    !git pull origin main
else:
    %cd /kaggle/working
    !git clone https://github.com/razz1992/cuda-cpp-gpu-lab.git
    %cd /kaggle/working/cuda-cpp-gpu-lab

print('Repository:', os.getcwd())
!git log -1 --oneline

## 3. Experiment 1 — vector addition correctness

In [ ]:
!nvcc -O2 -std=c++17 vector_add.cu -o vector_add
!./vector_add
!./vector_add 10000000

## 4. Experiment 1 — CPU vs GPU timing

In [ ]:
!nvcc -O2 -std=c++17 benchmark.cu -o benchmark
!./benchmark

In [ ]:
!./benchmark 1000000 100
!./benchmark 10000000 100
!./benchmark 50000000 30

## Experiment 1 — interpretation

- CPU average: repeated CPU vector addition.
- GPU kernel average: kernel-only time, excluding host/device copies.
- GPU end-to-end average: input transfers + kernel + output transfer.
- Compare CPU with GPU end-to-end for application-level latency. Small inputs can lose to CPU because launch and transfer overhead dominate.

# Experiment 2 — GPU memory optimization

Compare three approaches: A allocates/frees device memory each iteration; B reuses device allocations; C uses pinned host memory and asynchronous copies in one CUDA stream.

In [ ]:
!nvcc -O2 -std=c++17 memory_optimization.cu -o memory_optimization
!./memory_optimization

## 5. Run different workload sizes

Command format: `./memory_optimization [element_count] [iterations]`.

In [ ]:
!./memory_optimization 1000000 50
!./memory_optimization 10000000 50
!./memory_optimization 50000000 20

## 6. Key points to observe

- **A vs B:** isolates the impact of repeated `cudaMalloc` and `cudaFree`; B should benefit from reusing allocations.
- **B vs C:** explores page-locked (pinned) host memory and asynchronous transfer submission.
- All three approaches must report validation PASS.
- C queues the batch in one stream and synchronizes once at the end. A/B use blocking copies each iteration, so C is a batch-throughput comparison, not a perfectly identical per-iteration microbenchmark.
- Pinned memory is a limited system resource. Allocate only what is needed.
- The single stream serializes H2D → kernel → D2H. True overlap requires multiple independent buffers and a multi-stream/double-buffered design.

Repeat runs, record GPU model, element count, iteration count and timing. Kaggle sessions are ephemeral; save outputs you need.